# Nemotron-3-Nano LoRA SFT with CoT-Selected Training Data

## Approach

This notebook fine-tunes **Nemotron-3-Nano-30B-A3B-BF16** with LoRA using carefully curated Chain-of-Thought (CoT) training data.

The key insight is that **training data quality matters more than quantity**: rather than using all 9,000 training samples, I generated CoT reasoning via LLM inference and then **kept only the correct answers** verified by rule-based scoring.

### CoT Data Generation Pipeline

The training data (`train_split_with_cot.csv`) was generated by the following steps:

1. **Prompt construction**: Each puzzle prompt was augmented with the Kaggle official suffix: `Please put your final answer inside \boxed{}. For example: \boxed{your answer}`

2. **Type-specific prompt engineering**:
   - **Text Encryption**: Embedded the 77-word Alice's Wonderland dictionary into the prompt. Since these puzzles use monoalphabetic substitution ciphers with a fixed vocabulary, providing the encrypted→plain mapping for all 77 words (with `?` for unknown characters) dramatically improved accuracy.
   - **Bit Manipulation**: Embedded per-bit boolean function candidate analysis. Each output bit is independently determined by 1-3 input bits via boolean functions (ID 40%, AND 15%, XOR 8%, etc.). The solver enumerates all candidate functions for each bit and identifies CERTAIN vs AMBIGUOUS bits.
   - **Other types** (Gravitational Constant, Unit Conversion, Numeral Conversion, Equation Transformation): Standard step-by-step reasoning prompts.

3. **LLM inference** with temperature=0.7 to generate CoT reasoning

4. **Rule-based correctness filtering**: Extracted the final answer from `\boxed{}` and compared with ground truth using type-specific checkers:
   - Bit Manipulation: exact 8-bit binary match
   - Gravitational Constant / Unit Conversion: numeric match within ±0.05 absolute or ±0.5% relative error
   - Numeral Conversion: Roman numeral exact match (case-insensitive)
   - Text Encryption: case-insensitive exact match
   - Equation Transformation: exact string match
   
   **Only samples where the LLM's answer was verified correct were kept.**

5. **Token length control**: CoT outputs were checked against the Nemotron-3-Nano tokenizer. Any `generated_cot` exceeding **7,600 tokens** was sent back to the LLM with instructions to shorten by the overflow percentage while preserving the answer in `\boxed{}`. The limit of 7,600 (not 7,680) leaves headroom for the `</think>\n\boxed{answer}` suffix appended during training.

### Dataset Statistics

The resulting dataset contains **6,558 verified-correct CoT samples** out of 9,000 total:

| Puzzle Type | Total Available | Sampled for SFT | % Used | Note |
|---|---:|---:|---:|---|
| Gravitational Constant | 1,511 | 400 | 26.5% | Easy: 99.7% pass rate |
| Numeral Conversion | 1,491 | 300 | 20.1% | Easy: 99.6% pass rate |
| Unit Conversion | 1,342 | 700 | 52.2% | Good: 88.7% pass rate |
| Text Encryption | 1,407 | 700 | 49.8% | Good: 94.3% pass rate (with 77-word dict prompt) |
| Bit Manipulation | 607 | 607 | **100%** | Hard: only 40.3% pass rate — all correct samples used |
| Equation Transformation | 200 | 200 | **100%** | Hardest: only 13.6% pass rate — all correct samples used |
| **Total** | **6,558** | **2,907** | **44.3%** | |

Bit Manipulation and Equation Transformation had much lower pass rates because these puzzle types require complex pattern recognition that even strong LLMs struggle with. All verified-correct samples were used for these types.

### What I Did NOT Tune

My main effort was on **data preparation** (CoT generation + correctness filtering). I did **not** extensively tune the LoRA configuration — I used the standard competition target modules (`in_proj|out_proj|up_proj|down_proj`) with rank=32, alpha=32. If you want to improve accuracy further, tuning LoRA hyperparameters (alpha, dropout, learning rate, epochs) or trying different target module selections could help!

Oops, I forgot to say that I didn't input solvers into Equation Transformation, so we may get better scores from inputting "Equation Transformation" knowledge to solve.
(I've not yet analyzed "Equation Transformation" thingy...)



***First go to settings/accelerator --> GPU T4 x 2***

In [1]:
# Core Libraries
import os, glob, sys, subprocess, site, importlib.util, shutil, stat, types, re

import datasets
import kagglehub

import torch
# import mamba_ssm later

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

import pandas as pd
import random
import gc, time

import json, zipfile

print("Core Libraries imported")

Core Libraries imported


## Setup & Model Loading

In [2]:
candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)

print("Found Triton wheels:", candidates)

if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]

target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps",
        "--target", target,
        "--upgrade",
        "--ignore-installed",
        wheel,
    ],
    check=True,
)

if target not in sys.path:
    sys.path.insert(0, target)

site.addsitedir(target)

print("Custom target added:", target)
print("triton spec:", importlib.util.find_spec("triton"))

Found Triton wheels: ['/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl', '/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/triton-3.5.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl', '/kaggle/input/datasets/mayukh18/nemotron-packages/packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl']
Processing /kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
Custom target added: /kaggle/working/pydeps
triton spec: ModuleSpec(name='triton', loader=<_frozen_importlib_external.SourceFileLoader object at 0x79439a89fce0>, origin='/usr/local/lib/python3.12/dist-packages/triton/__init__.py', submodule_search_locations=['/usr/local/lib/python3.12/dist-packages/triton'])


**Short Summary** of the script:

- Looks for a Triton wheel.

- Installs it manually.

- Adds it to the current environment.

- Verifies that it can be imported.

All of this without modifying Kaggle's global system.
     
    
**What is triton?**
Language/Library used to create ultra-fast GPU operations for AI without having to program in pure CUDA.


In [3]:
# Add utility script to Python path (provides helper binaries)
sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')

# Copy ptxas-blackwell to /tmp with execute permissions
ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
ptxas_dst = '/tmp/ptxas-blackwell'

In [4]:
if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
    shutil.copy2(ptxas_src, ptxas_dst)
    os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    src_bin = os.path.dirname(ptxas_src)
    dst_bin = '/tmp/triton_nvidia_bin'
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst

    import triton.backends.nvidia as nv_backend
    nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
    os.environ['TRITON_PTXAS_PATH'] = ptxas_dst

In [5]:
import triton.backends.nvidia.compiler as nv_compiler

nv_compiler.get_ptxas_version = lambda arch: '12.0'

print('Training environment fixes applied.')


Training environment fixes applied.


This block prepares and "patches" the Kaggle GPU environment so that OpenAI Triton can compile kernels correctly during training.          
 
**What problem attempts to fix ?**  

On Kaggle, sometimes: CUDA binaries are missing, Triton cannot find ptxas, or there are incompatibilities with newer GPUs (Blackwell/Hopper/etc).    
     
Therefore, this code uses "hacks" to force Triton to work.

In [6]:
# Install trl if needed (for training mode)
try:
    import trl
    print(f"trl already installed: {trl.__version__}")
    
except ImportError:
    # Try offline install first, then online
    offline_path = "/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/"
    
    if os.path.exists(offline_path):
        subprocess.run(f"pip install --no-index --find-links={offline_path} trl", shell=True)
    else:
        subprocess.run("pip install trl", shell=True)
        
    import trl
    print(f"trl installed: {trl.__version__}")

Looking in links: /kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/
Processing /kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/trl-0.29.1-py3-none-any.whl
trl installed: 0.29.1


Short summary of the script:

Checks if trl (**Transformer Reinforcement Learning**) exists.

If it is not found, installs it.

In [7]:
def sh(cmd: str, check: bool = True):
    print("+", cmd)
    return subprocess.run(cmd, shell=True, check=check)

def find_spec(name: str) -> bool:
    return importlib.util.find_spec(name) is not None

def recursive_wheels(pattern: str):
    return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

In [8]:
all_mamba = recursive_wheels("mamba_ssm-*.whl")
all_causal = recursive_wheels("causal*conv1d*.whl")
all_datasets = recursive_wheels("datasets-*.whl")
all_trl = recursive_wheels("trl-*.whl")
all_multiprocess = recursive_wheels("multiprocess-*.whl")
all_dill = recursive_wheels("dill-*.whl")
all_xxhash = recursive_wheels("xxhash-*.whl")

print("Found mamba wheels:", all_mamba)
print("Found causal-conv1d wheels:", all_causal)

Found mamba wheels: ['/kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl']
Found causal-conv1d wheels: ['/kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl']


In [9]:
# About torch

py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
torch_mm = ".".join(torch.__version__.split("+")[0].split(".")[:2])
abi_tag = "cxx11abiTRUE" if torch.compiled_with_cxx11_abi() else "cxx11abiFALSE"

print("Python:", sys.version)
print("Torch: ", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA:", torch.version.cuda)
print("Wheel selector:", {"py_tag": py_tag, "torch": torch_mm, "abi": abi_tag})

if not torch.cuda.is_available():
    raise RuntimeError("Requires GPU runtime because mamba_ssm wheel is CUDA-based.")

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch:  2.10.0+cu128
CUDA available: True
Torch CUDA: 12.8
Wheel selector: {'py_tag': 'cp312', 'torch': '2.10', 'abi': 'cxx11abiTRUE'}


In [10]:
def pick_best(wheels):
    exact = [w for w in wheels if py_tag in w and f"torch{torch_mm}" in w and abi_tag in w]
    if exact:
        return exact[-1]
    py_only = [w for w in wheels if py_tag in w]
    if py_only:
        return py_only[-1]
    return None

In [11]:
if not find_spec("datasets"):
    w = pick_best(all_datasets)
    if w:
        sh(f'{sys.executable} -m pip install --no-index --no-deps "{w}"')
        
if not find_spec("trl"):
    w = pick_best(all_trl)
    if w:
        sh(f'{sys.executable} -m pip install --no-index --no-deps "{w}"')
        
for pkg, wheels in [("multiprocess", all_multiprocess), ("dill", all_dill), ("xxhash", all_xxhash)]:
    if not find_spec(pkg):
        w = pick_best(wheels)
        if w:
            sh(f'{sys.executable} -m pip install --no-index --no-deps "{w}"', check=False)

if not find_spec("mamba_ssm"):
    causal_wheel = pick_best(all_causal)
    mamba_wheel = pick_best(all_mamba)

    print("Selected causal wheel:", causal_wheel)
    print("Selected mamba wheel:", mamba_wheel)

    if causal_wheel:
        sh(f'{sys.executable} -m pip install --no-index --no-deps "{causal_wheel}"')
    if mamba_wheel:
        sh(f'{sys.executable} -m pip install --no-index --no-deps "{mamba_wheel}"')
    else:
        raise FileNotFoundError(f"No compatible mamba_ssm wheel found under /kaggle/input for "
                                f"py={py_tag}, torch={torch_mm}, abi={abi_tag}.")


Selected causal wheel: /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
Selected mamba wheel: /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
+ /usr/bin/python3 -m pip install --no-index --no-deps "/kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
Processing /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
+ /usr/bin/python3 -m pip install --no-index --no-deps "/kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
Processing /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl


In [12]:
for _mod_name in ['mamba_ssm.modules.mamba3',
                  'mamba_ssm.ops.cute',
                  'mamba_ssm.ops.cute.mamba3',
                  'mamba_ssm.ops.cute.mamba3.mamba3_step_fn']:
    
    sys.modules[_mod_name] = types.ModuleType(_mod_name)
sys.modules['mamba_ssm.modules.mamba3'].Mamba3 = None

In [13]:
import mamba_ssm

print(f'datasets:  {datasets.__version__}')
print(f'trl:       {trl.__version__}')
print(f'mamba_ssm: {mamba_ssm.__version__}')

datasets:  4.8.3
trl:       0.29.1
mamba_ssm: 2.3.1


This block:

- Detects your CUDA/PyTorch environment.

- Looks for compatible wheels.

- Installs dependencies offline.

- Installs mamba_ssm.

- Applies compatibility hacks.

- Sets up the training environment.

In [14]:
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")

print(f"Model path: {MODEL_PATH}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_PATH,
                                             device_map="auto",
                                             trust_remote_code=True,
                                             dtype=torch.bfloat16)
print("Model loaded.")


Model path: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

Model loaded.


Downloads Nemotron, loads the tokenizer, loads the LLM into the GPU, and prepares it for training/inference.

In [15]:
LORA_RANK = 32
LORA_ALPHA = 32

In [16]:
lora_config = LoraConfig(r=LORA_RANK,
                         lora_alpha=LORA_ALPHA,
                         target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
                         lora_dropout=0.05,
                         bias="none",
                         task_type=TaskType.CAUSAL_LM)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 880,138,240 || all params: 32,458,075,584 || trainable%: 2.7116


## What this block does (LoRA with PEFT)

This approach does not train the entire model. Instead, it uses **Hugging Face PEFT**, which allows for efficient fine-tuning without modifying all the LLM's parameters.

### 1. Freezes the base model
The original LLM remains intact (it is not trained).

### 2. Adds LoRA
Only small adaptation layers are incorporated:

- r = 32 → adaptation size (tuning capacity)
- alpha = 32 → update scale
    
This defines how much "flexibility" the adapters have.

### 3. Selection of layers to train

**target_modules** = r".*.(in_proj|out_proj|up_proj|down_proj)$"   
   
Only specific parts of the transformer are trained:
   
- Attention projections.
- Key transformation layers.

### 4. LoRA Configuration

lora_config = LoraConfig(...)

Includes:

- dropout = 0.05 → helps prevent overfitting
- task_type = CAUSAL_LM → GPT-style model (text generation)

### 5. Application to the model
The base model is converted into: *frozen model + trainable LoRA adapters*

### 6. Efficient training
Only small adapter layers are trained, not the full model.

## Mode A: Train on Kaggle

In [17]:
from datasets import Dataset as HFDataset
from trl import SFTTrainer, SFTConfig

SEED = 123
PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

In [18]:
# --- Dataset path ---

DATASET_PATH = "/kaggle/input/datasets/konbu17/nemotron-sft-lora-cot-selection/train_split_with_cot.csv"
df = pd.read_csv(DATASET_PATH)

print(f"Full dataset: {len(df)} rows")
print(df["type"].value_counts().sort_index())

Full dataset: 6558 rows
type
Bit Manipulation            607
Equation Transformation     200
Gravitational Constant     1511
Numeral Conversion         1491
Text Encryption            1407
Unit Conversion            1342
Name: count, dtype: int64


In [19]:
# --- Type-based sampling ---

TYPE_SAMPLES = {"Numeral Conversion": 300,
                "Gravitational Constant": 400,
                "Unit Conversion": 700,
                "Text Encryption": 700,
                "Bit Manipulation": 607,         # all available
                "Equation Transformation": 200}  # all available
            
sampled_dfs = []

for ptype, n_samples in TYPE_SAMPLES.items():
    subset = df[df["type"] == ptype]
    
    if n_samples >= len(subset):
        sampled = subset
    else:
        sampled = subset.sample(n=n_samples, random_state=SEED)
        
    print(f"  {ptype}: {len(subset)} -> {len(sampled)}")
    
    sampled_dfs.append(sampled)

train_df = pd.concat(sampled_dfs, ignore_index=True)
train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"\nTraining samples: {len(train_df)}")

  Numeral Conversion: 1491 -> 300
  Gravitational Constant: 1511 -> 400
  Unit Conversion: 1342 -> 700
  Text Encryption: 1407 -> 700
  Bit Manipulation: 607 -> 607
  Equation Transformation: 200 -> 200

Training samples: 2907


In [20]:
# --- Build SFT dataset ---

records = []

for _, row in train_df.iterrows():
    prompt = str(row["prompt"])
    answer = str(row["answer"])
    cot = str(row["generated_cot"])
    
    if not cot or cot == "nan" or len(cot.strip()) < 5:
        continue
        
    cot_cleaned = re.sub(r'\\boxed\{[^}]*\}', '', cot).rstrip()
    user_content = prompt + PROMPT_SUFFIX
    
    # chat template auto-adds <think>\n, so assistant starts with CoT directly
    assistant_content = cot_cleaned + f"\n</think>\n\\boxed{{{answer}}}"
    
    records.append({"messages": [{"role": "user", "content": user_content},
                                 {"role": "assistant", "content": assistant_content}]})
    
dataset = HFDataset.from_list(records)

print(f"SFT records: {len(records)}")

SFT records: 2907


In [21]:
# --- Training Args ---

training_args = SFTConfig(output_dir="/kaggle/working/sft_output",
                          num_train_epochs=2, # 1 # 2
                          per_device_train_batch_size=1,
                          gradient_accumulation_steps=8,
                          learning_rate=1e-4, # 5e-5 # 1e-4
                          lr_scheduler_type="cosine",
                          warmup_ratio=0.05,
                          max_length=4096, # 7680 # 4096
                          logging_steps=10,
                          save_strategy="no",
                          bf16=True,
                          gradient_checkpointing=True,
                          gradient_checkpointing_kwargs={"use_reentrant": False},
                          dataloader_num_workers=2,
                          remove_unused_columns=False,
                          seed=SEED,
                          report_to="none",
                          packing=False)

trainer = SFTTrainer(model=model,
                     args=training_args,
                     train_dataset=dataset,
                     processing_class=tokenizer)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/2907 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2907 [00:00<?, ? examples/s]

In [22]:
# --- Training ---

print("Starting SFT training...")

t0 = time.time()
trainer.train()
elapsed = time.time() - t0

print(f"Training done in {elapsed/60:.1f} min")

# Save adapter
ADAPTER_DIR = "/kaggle/working/sft_adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Adapter saved to {ADAPTER_DIR}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 11, 'pad_token_id': 11}.


Starting SFT training...


Step,Training Loss
10,10.748180
20,9.176211
30,7.534019
40,5.025960
50,4.736396
60,3.279167
70,3.528776
80,3.259951
90,3.050975
100,3.401916


Training done in 410.3 min
Adapter saved to /kaggle/working/sft_adapter


# 🧠 MAIN TRAINING CODE
It trains the model using verified Chain-of-Thought (CoT) examples, but only using LoRA (not the entire model).

### 1. Loading and preparing the dataset

Loads a dataset with:

- prompt (problem)

- answer (correct answer)

- generated_cot (reasoning)

### 2. Displaying type distribution

Used to see how many examples exist per puzzle type.

### 3. Sampling by type (balancing)

Selects how many examples to use from each category:

- Easy → few or moderate

- Hard → all available

Example:

- Bit Manipulation → 100% (607)

- Equation Transformation → 100% (200)

This prevents the dataset from being unbalanced.

### 4. Dataset shuffling

Shuffles the data to avoid biased ordering.

### 5. Construction of the training dataset

Converts each example into chat format:

- Input (user) prompt + instruction: "Please put your final answer inside \boxed{}"

- Output (assistant) Clean CoT +  + \boxed{answer}

This process:

- Removes \boxed{} from the intermediate CoT

- Forces a structured format: reasoning + final answer

### 6. Hugging Face Dataset

Converts everything into a training-compatible format.

### 7. Training configuration

Here you define how to train, **Key parameters:**

1. num_train_epochs=2 → 2 passes through the dataset

2. batch_size=1 → small due to GPU memory

3. gradient_accumulation_steps=8 → simulates a larger batch

4. learning_rate=1e-4 → learning speed

5. max_length=4096 → maximum sequence length

6. bf16=True → optimized precision

7. gradient_checkpointing=True → saves VRAM

### 8. SFT Trainer

Hugging Face TRL (Transformer Reinforcement Learning)to perform: supervised fine-tuning on conversations.

### 9. Actual training

This is where the important part happens: the model learns the CoT patterns and adjusts only LoRA (not the whole model)

### 10. Saving the model

## Create submission.zip

In [23]:
OUTPUT_DIR = "/kaggle/working"
SUBMISSION_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "submission_adapter")
os.makedirs(SUBMISSION_ADAPTER_DIR, exist_ok=True)

required_files = ["adapter_config.json", "adapter_model.safetensors"]

src_adapter_dir = "/kaggle/working/sft_adapter"
print("Packaging freshly trained adapter from:", src_adapter_dir)


Packaging freshly trained adapter from: /kaggle/working/sft_adapter


In [24]:
for fname in required_files:
    src = os.path.join(src_adapter_dir, fname)
    dst = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing required adapter file: {src}")
    shutil.copy2(src, dst)
    print(f"Copied {fname} ({os.path.getsize(dst)/1024/1024:.1f} MB)")

config_path = os.path.join(SUBMISSION_ADAPTER_DIR, "adapter_config.json")

with open(config_path, "r") as f:
    cfg = json.load(f)
    
BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"] = 0.0

with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

zip_path = os.path.join(OUTPUT_DIR, "submission.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        fpath = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
        zf.write(fpath, fname)
        print(f"  Added {fname}")

zip_sz = os.path.getsize(zip_path) / 1024 / 1024

print(f"\nsubmission.zip: {zip_sz:.1f} MB")
print("Done! Ready to submit.")


Copied adapter_config.json (0.0 MB)
Copied adapter_model.safetensors (3359.2 MB)
  Added adapter_config.json
  Added adapter_model.safetensors

submission.zip: 3084.9 MB
Done! Ready to submit.
